# Planet basemap parcel statistics

Process PL_BSM tile groups sequentially with the shared cleaning, parcel statistics,
quality metrics and output format. Download only one month for the current group,
read cropped pixels, and delete those downloads before the next month. Cross-tile
parcels use a mosaic of all their intersecting quads and produce one result each.


In [ ]:
import os
from pathlib import Path

from common_libraries.io_library import read_data
from common_libraries.catalog_library import CatalogSearchUtils
from hub_catalog.planet_parcel_stats import PlanetBasemapZonalStats


In [ ]:
WORK_PATH = Path("C:/work_dir")
RUN_IDENTIFIER = "volvi-2023-2024"
PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "parcels_volvi.gpkg"
BSM_TILES_PATH = WORK_PATH / "planet_basemap_quads.gpkg"
SCRATCH_PATH = WORK_PATH / RUN_IDENTIFIER / "planet_scratch"
OUTPUT_DIR = WORK_PATH / RUN_IDENTIFIER / "1_planet_parcel_stats"
START_DATE = "2023-10-01"
END_DATE = "2024-09-30"
WORKING_EPSG = 2100
PARCEL_ID_FIELD = "parcel_code"
QUAD_ID_FIELD = "quad_id"
COLLECTION_ID = "PL_BSM"
ANALYSIS_PATTERN = "*Analysis*.tif"  # Adjust only if catalog filenames differ.

# NDMI requires SWIR, which Planet eight-band tiles do not contain.
PLANET_INDICES = ["EVI", "NDRE", "NDVI", "NDWI", "SAVI"]
SPATIAL_STATISTICS = ["mean", "median", "sd", "min", "max", "range"]
REFLECTANCE_SCALE = 0.0001  # Fallback for DN scaled by 10000; use 1.0 for reflectance already in 0-1 units.


## Parcels and intersecting quads
CRS metadata is required. The extractor repairs polygon geometries while preserving
multipart parcels, identifiers and source attributes. Do not explode multipart
parcels into duplicate IDs. Copy any label enrichment from your original notebook
here if those labels are not already stored in the parcel file.


In [ ]:
gdf_parcels = read_data(str(PARCELS_PATH))
gdf_tiles = read_data(str(BSM_TILES_PATH))
if gdf_parcels.crs is None or gdf_tiles.crs is None:
    raise ValueError("Parcels and tile grid must have explicit CRS metadata")
gdf_tiles = gdf_tiles.to_crs(WORKING_EPSG)
extractor = PlanetBasemapZonalStats(
    parcels=gdf_parcels, start_date=START_DATE, end_date=END_DATE,
    output_dir=OUTPUT_DIR, working_epsg=WORKING_EPSG,
    parcel_id_field=PARCEL_ID_FIELD,
    resolution_metres=5, batch_size_metres=2000, buffer_metres=100,
    max_cube_bytes=512_000_000,
    indices=PLANET_INDICES, reflectance_scale=REFLECTANCE_SCALE,
    spatial_statistics=SPATIAL_STATISTICS,
    remove_outliers=True, iqr_quantiles=(0.1, 0.9), iqr_multiplier=1.5,
    iqr_min_valid_pixels=20, fill_nulls=True,
    temporal_fill_mode="bidirectional",  # Matches 1_parcel_stats; use past_only for causal features.
    minimum_parcel_pixels=3, minimum_observed_fraction_for_fill=0.20,
    spatial_fill_window_sizes=(3, 5), interpolation_method="nearest",
    interpolation_max_distance_in_meters=500,
)
# Select tiles intersecting the parcels; run_catalog assigns each parcel once.
study_area = extractor.parcels.to_crs(WORKING_EPSG).geometry.union_all()
gdf_tiles = gdf_tiles[gdf_tiles.intersects(study_area)].copy()
if gdf_tiles.empty:
    raise ValueError("No Planet quads intersect the study area")
print(f"{len(extractor.parcels)} parcels; {len(gdf_tiles)} quads")


## Connect to the catalog
This cell only connects. The extraction cell below downloads tiles on demand.
Missing or ambiguous quad/month items stop the batch with an explicit error.


In [ ]:
catalog = CatalogSearchUtils(
    catalog_endpoint=os.environ["CATALOG_URL"],
    client_id=os.getenv("CATALOG_CLIENT_ID", "internal"), verbose=True,
)


## Calculate parcel statistics
Tiles are aligned to a common 5 m grid. The first eight TIFF bands follow Planet
order: coastal blue, blue, green I, green, yellow, red, red edge, NIR.
TIFF nodata and masks are respected. TIFF scale/offset metadata takes precedence;
otherwise REFLECTANCE_SCALE converts stored values to reflectance before cleaning.
Use 0.0001 for DN scaled by 10000, or 1.0 for tiles already in reflectance units.
Physical-band statistics are consequently in reflectance units.

Indices are calculated per pixel after cleaning, then each requested statistic
is calculated over parcel pixels:
- EVI = 2.5 * (B8 - B6) / (B8 + 6 * B6 - 7.5 * B2 + 1)
- NDRE = (B8 - B7) / (B8 + B7)
- NDVI = (B8 - B6) / (B8 + B6), replacing the TIFF ninth band
- NDWI = (B4 - B8) / (B4 + B8), matching the existing green/NIR definition
- SAVI = 1.5 * (B8 - B6) / (B8 + B6 + 0.5)

Zero denominators become null. NDMI = (NIR - SWIR) / (NIR + SWIR) cannot
be calculated from these tiles because they lack SWIR. Requesting it raises an
explicit error rather than substituting another band.

[Planet band order and reflectance scaling](https://support.planet.com/hc/en-us/articles/115013886368-How-is-the-Surface-Reflectance-Product-Provided)

IQR cleaning and filling use each buffered processing batch. Batch size, resolution
and buffer affect results. Cleaning needs several times the raw cube RAM; reduce
batch size if necessary. Downstream feature selection must use Planet band names.

### Disk usage and processing order
1. Process parcels contained in each single quad.
2. Process remaining parcels grouped by their intersecting quad IDs.
3. Split each group into spatial batches to bound cropped cube memory.
4. For each month, download only the analysis TIFFs for that group, mosaic their
   cropped pixels on the batch grid, then delete the full TIFFs.
5. Clean the cropped time series, calculate statistics, and commit the small
   batch result and cleaning report. Delete temporary raw and cleaned cubes.

Temporal filling still sees all months. Peak temporary storage is one month's
analysis tiles for the current group, or the current batch's intermediate cubes.
The buffer provides context inside the selected quad(s); no extra quads are
fetched solely for the buffer. A quad may be downloaded again for another spatial
batch or cross-tile group to keep disk usage bounded.
Cleanup affects only unique subdirectories created under SCRATCH_PATH, including
on Python exceptions. A hard process termination can leave a scratch directory.


In [ ]:
merged_gdf = extractor.run_catalog(
    catalog, gdf_tiles, scratch_dir=SCRATCH_PATH,
    collection_id=COLLECTION_ID, quad_id_field=QUAD_ID_FIELD,
    analysis_pattern=ANALYSIS_PATTERN,
    resume=True,  # Reuse completed batch tables; False refreshes catalog imagery.
)
geoparquet_path = OUTPUT_DIR / extractor.PARCEL_OUTPUT_FILE_NAME
print(f"Saved {len(merged_gdf)} parcels and {len(merged_gdf.columns)} columns: {geoparquet_path}")
merged_gdf.head()


## Outputs
- `satellite_parcel_time_stats.geoparquet`: one row per parcel, source attributes,
  geometry, quality metrics and dated index/band statistics.
- `satellite_parcel_annual_stats.geoparquet`: shared annual median feature reduction.
- `satellite_pixel_cleaning_report.csv`: cleaning audit by batch, month and band.
- `planet_tile_manifest.csv`: catalog identifiers and deleted temporary source paths.
- `streaming_results/`: small batch result/report checkpoints for restart.

No full tiles or intermediate raster cubes are retained after successful processing.
Re-run the extraction cell to resume from committed batch tables. Use resume=False
if imagery in the catalog changed. Existing downloads from previous workflows are
not touched. Outputs remain separate from the Sentinel run.
